# Strictly nested meta-inference: admitted or not


**The admission test (instruction 64.34).** Label `compartment`. 5 outer folds of whole orthogroups; inside each, the base methods' calls on the training genes come from 3 inner folds that never see an outer-test label, and their calls on the test genes from models fitted with the test labels hidden. The meta-model learns from the inner calls only and is scored once on the outer-test genes, against the best single method chosen on the training genes, the plurality vote and the commonest class.

Built 2026-11-07.

In [1]:
import os, subprocess, sys
from concurrent.futures import ThreadPoolExecutor
import numpy as np, pandas as pd
from sklearn.linear_model import LogisticRegression
from scipy.stats import binomtest
from parascape import track_record as T
KEYS = T.supported('Tg')
SCRIPT = os.path.join(REPO, 'scripts', 'build_meta_inference_nested_notebook.py')
def run(key):
    out = os.path.join(OUT, f'calls_{key}.parquet')
    if not os.path.exists(out):
        subprocess.run([sys.executable, SCRIPT, '--strategy', key, '--target', TARGET], check=False,
                       env={**os.environ, 'OMP_NUM_THREADS': '2'})
    return pd.read_parquet(out).assign(strategy=key) if os.path.exists(out) else None
with ThreadPoolExecutor(max_workers=7) as pool:
    parts = [p for p in pool.map(run, KEYS) if p is not None and len(p)]
calls = pd.concat(parts, ignore_index=True)
print(calls.strategy.nunique(), 'base methods;', calls.gene_id.nunique(), 'genes')

14 base methods; 3804 genes


In [2]:
rows, paired = [], dict(meta_only=0, best_only=0)
for k in sorted(calls.outer_fold.unique()):
    c = calls[calls.outer_fold == k]
    tr = c[c.role == 'inner_train'].pivot_table(index='gene_id', columns='strategy', values='call', aggfunc='first').fillna('ABSTAIN')
    te = c[c.role == 'outer_test'].pivot_table(index='gene_id', columns='strategy', values='call', aggfunc='first').fillna('ABSTAIN')
    cols = sorted(set(tr.columns) & set(te.columns)); tr, te = tr[cols], te[cols]
    ytr = c[c.role == 'inner_train'].groupby('gene_id').truth.first().reindex(tr.index)
    yte = c[c.role == 'outer_test'].groupby('gene_id').truth.first().reindex(te.index)
    Xtr = pd.get_dummies(tr.astype(str)); Xte = pd.get_dummies(te.astype(str)).reindex(columns=Xtr.columns, fill_value=0)
    meta = LogisticRegression(max_iter=3000).fit(Xtr.astype(float), ytr)
    pm = meta.predict(Xte.astype(float))
    best = max(cols, key=lambda m: float((tr[m] == ytr).mean()))
    pb = te[best].to_numpy()
    pv = te.apply(lambda r: r[r != 'ABSTAIN'].value_counts().index[0] if (r != 'ABSTAIN').any() else 'ABSTAIN', axis=1).to_numpy()
    y = yte.to_numpy()
    for name, p in (('meta-model', pm), ('best single', pb), ('plurality vote', pv), ('commonest class', np.full(len(y), ytr.value_counts().index[0]))):
        rows.append(dict(fold=int(k), method=name, chosen=best if name == 'best single' else '', accuracy=float((p == y).mean()), n=len(y)))
    paired['meta_only'] += int(((pm == y) & (pb != y)).sum()); paired['best_only'] += int(((pb == y) & (pm != y)).sum())
res = pd.DataFrame(rows)
summary = res.groupby('method').agg(accuracy=('accuracy', 'mean'), low=('accuracy', 'min'), high=('accuracy', 'max')).round(3).sort_values('accuracy', ascending=False)
p = binomtest(paired['meta_only'], paired['meta_only'] + paired['best_only'], 0.5).pvalue
admitted = bool(summary.loc['meta-model', 'accuracy'] > summary.loc['best single', 'accuracy'] and p < 0.05)
res.to_csv(os.path.join(OUT, 'folds.tsv'), sep='\t', index=False); summary.to_csv(os.path.join(OUT, 'summary.tsv'), sep='\t')
print('paired:', paired, '; McNemar p', round(p, 5), '; admitted:', admitted)
summary

paired: {'meta_only': 460, 'best_only': 311} ; McNemar p 0.0 ; admitted: True


,accuracy,low,high
method,,,
meta-model,0.543,0.52,0.583
plurality vote,0.507,0.488,0.547
best single,0.504,0.481,0.537
commonest class,0.202,0.177,0.216
